# NB08_fPINN — Constrained multibody plant of the delta robot: does the fPINN still help, and with a non-Caputo memory?

**Purpose.** Benchmark B of the article. In NB01–NB04 the data came from the **same reduced 14-parameter model** the networks use, plus a synthetic torque. Here the data and the closed loop come from an independent **constrained multibody model** of the delta robot, and the reduced model is only the nominal model of the identifier and controller:

- **Plant.** Generalized coordinates x = (q, p): three arm angles and the platform position, three distance constraints |p − C_i(q_i)|² = L_B² (closed chains), Lagrange multipliers and Baumgarte stabilization. Arms with their own inertia (arm + motor + gearbox) and first moment, elbow masses, parallelogram forearms as **slender rods with distributed mass** (exact kinetic energy m/6·(v₁² + v₁·v₂ + v₂²)), translating platform with payload. Gravity along +z (z points down).
- **Reduced nominal model.** The 14-parameter regressor of the paper with γ(m) = (m₀ + m)/m₀, **identified by least squares on the multibody plant** (θ₀, m₀). Its residual is the structural gap the networks must learn.
- **Two joint-dissipation scenarios.**
  - `caputo`: viscous + Coulomb joint friction and a Caputo damper B·D^α q (α = 0.5, B = 0.05 N m s^α), as in the paper.
  - `lugre`: LuGre joint friction (bristle state with presliding hysteresis and Stribeck effect), **no** Caputo term. A physical memory that is not fractional: does the fPINN still help, and does its B̂ stay below the detection threshold of NB04 (0.016)?
- **Networks.** PINN and fPINN of NB02/NB04 (same architecture, training protocol and fast batches), 175 trajectories, seeds 0–2, per scenario: 12 networks.
- **Closed loop.** CTC + I of NB03 (same gains, 5 N m limit, 2 ms memory update) on the multibody plant: Adept-like cycle at the three speed scales × payload 0 / 1 kg; nominal reduced model, oracle (exact multibody model + true friction and memory) and the six networks per scenario.
- **Checks for the paper.** Energy conservation and constraint drift of the free multibody system, replay of inverse-dynamics torques through the forward dynamics, structural gap of the reduced model, energy dissipated by the true and identified memory torque.

**Payload.** The multibody plant carries the payload physically (on the platform), so 6 kg would exceed the 5 N m drives. The trajectory generator keeps its random stream and draws 0–6 as before; NB08 maps the draw to 0–1 kg (`PAYLOAD_SCALE = 1/6`).

**Memory term.** Identifiers and controllers use the full-memory L1 scheme only, as in NB01–NB04.

**One switch.** `QUICK = True` checks the whole pipeline in about 3 min (20 trajectories, 2 epochs, one seed, closed loop at the middle speed). `QUICK = False` runs **everything above** with nothing else to edit.

**Expected runtime (Ryzen 7 5850U, 16 GB): about 1.5 h, between 1 and 2.5 h (≈ 40 min of training for 12 networks, measured here at 0.2 s per epoch with 20 trajectories and scaled to 175; ≈ 20 min of closed loop, 96 runs of 3–20 s; ≈ 3 min of data generation). ** Every finished network is cached in `CACHE_DIR` with its weights in `WEIGHTS_OUT`, and every closed-loop run is appended to the CSV, so an interruption loses at most the network or run in progress: rerun and it resumes. The notebook prints its own estimate after every network.

**Nothing else needed in the folder:** NB08 does not read any earlier cache.

**Outputs:** `NB08_fPINN_Verificacion.csv`, `NB08_fPINN_ModeloReducido.csv`, `NB08_fPINN_TrainingLog.csv`, `NB08_fPINN_ClosedLoop.csv`, `NB08_fPINN_Summary.csv`, `fig_nb08_fpinn.png`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/Anaconda: avoids "OMP Error #15" when NumPy (MKL) and PyTorch both load OpenMP
import gc, time, copy, random, pickle, numpy as np, pandas as pd
from math import gamma as Gamma
from scipy.signal import fftconvolve
import torch, torch.nn as nn
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt

# ---- settings ---------------------------------------------------------------------------------
QUICK         = True               # True: fast check (~3 min). False: every experiment of NB08 (see header).
B_FRAC        = 0.05               # [N m s^α] Caputo damper per joint (scenario caputo)
ALPHA_MAIN    = 0.5
FRICTION      = dict(b_v=0.05, f_c=0.04, eps=0.02)                     # viscous + Coulomb joint friction (scenario caputo)
LUGRE         = dict(s0=40.0, s1=0.2, s2=0.05, Fc=0.04, Fs=0.06, vs=0.1) # LuGre per joint (scenario lugre): N m/rad, N m s/rad, N m s/rad, N m, N m, rad/s
MBP           = dict(I_A=0.050, mr_A=0.028, m_E=0.050, m_F=0.120, m_P=0.250)
                   # arm + motor + gearbox inertia about the joint [kg m²], arm first moment [kg m], elbow mass [kg],
                   # parallelogram forearm (two rods) [kg], platform [kg]
PAYLOAD_SCALE = 1/6                # generator draw 0..6 -> 0..1 kg
G_ACC         = 9.81
BAUMGARTE_W   = 200.0              # [rad/s] Baumgarte stabilization, critically damped
DETECT_B      = 0.016              # NB04 null-test threshold on B̂
TRAIN_CFG     = dict(batch_size=2048, lr=1e-3, weight_decay=1e-5, max_epochs=150,
                     patience=18, lr_patience=6, log_every=25)               # NB02 / NB04
ALPHA_INIT    = 0.8
CACHE_DIR     = "NB08_fPINN_cache"
WEIGHTS_OUT   = "NB08_fPINN_weights"
SPEED_SCALES, PAYLOADS = (6.4, 3.2, 1.6), (0.0, 1.0)
SCENARIOS     = ("caputo", "lugre")

if not QUICK:
    N_TRAIN_MAX = 175
    EXPERIMENTS = [(scn, k, 175, s) for scn in SCENARIOS for s in (0, 1, 2) for k in ("PINN", "fPINN")]
else:
    N_TRAIN_MAX = 20; SPEED_SCALES, PAYLOADS = (3.2,), (0.0,)
    TRAIN_CFG = dict(TRAIN_CFG, max_epochs=2, log_every=1)
    EXPERIMENTS = [(scn, k, 20, 0) for scn in SCENARIOS for k in ("PINN", "fPINN")]
E = TRAIN_CFG["max_epochs"]; SUF = "_QUICK" if QUICK else ""    # QUICK writes apart, so it never mixes with the full run
# -----------------------------------------------------------------------------------------------
FS = 2000; dt = 1.0/FS; SUBSAMPLE = 5
DATA_SEED, CAL_SEED, N_VAL_TRAJ, N_TEST_TRAJ, N_CAL_TRAJ = 12345, 2024, 20, 20, (60 if not QUICK else 15)
ALPHA_GRID = np.round(np.arange(0.05, 1.0001, 0.025), 3); NA = len(ALPHA_GRID)
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(WEIGHTS_OUT, exist_ok=True)
torch.set_num_threads(min(8, max(1, (os.cpu_count() or 2)//2)))
print("QUICK" if QUICK else "FULL", f"| {len(EXPERIMENTS)} networks | epochs {E} | torch {torch.__version__} | threads {torch.get_num_threads()}")

## Kinematics and reduced regressor (NB01–NB04)

In [ ]:
R_BASE, R_PLAT, L_A, L_B = 0.114, 0.035, 0.200, 0.460

def ik_vec(P):
    x, y, z = P[:, 0], P[:, 1], P[:, 2]; q = np.zeros((len(P), 3)); rd = R_BASE - R_PLAT
    for i in range(3):
        al = 2*i*np.pi/3; u = -2*L_A*z; aux = x*np.cos(al) + y*np.sin(al)
        v = 2*L_A*(rd - aux); w = L_B**2 - (rd**2 - 2*rd*aux + x**2 + y**2 + z**2 + L_A**2)
        r = np.sqrt(u**2 + v**2 - w**2); den = u**2 + v**2
        s1 = (u*w + v*r)/den; c1 = (v*w - u*r)/den; t1 = np.where(c1 >= 0, np.arcsin(s1), np.pi - np.arcsin(s1))
        s2 = (u*w - v*r)/den; c2 = (v*w + u*r)/den; t2 = np.where(c2 >= 0, np.arcsin(s2), np.pi - np.arcsin(s2))
        q[:, i] = np.where(np.abs(t1) < np.abs(t2), t1, t2)
    return q

def fk_vec(q):
    rd = R_BASE - R_PLAT; C = np.zeros((len(q), 3, 3))
    for i in range(3):
        a = 2*i*np.pi/3; r = rd + L_A*np.cos(q[:, i])
        C[:, i] = np.column_stack([r*np.cos(a), r*np.sin(a), L_A*np.sin(q[:, i])])
    A1, B1, C1 = (2*(C[:, 1] - C[:, 0])).T; D1 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 1]))
    A2, B2, C2 = (2*(C[:, 2] - C[:, 0])).T; D2 = 2*L_A*rd*(np.cos(q[:, 0]) - np.cos(q[:, 2]))
    den = A2*B1 - A1*B2
    L1 = (A1*D2 - A2*D1)/den; L2 = (A1*C2 - A2*C1)/den; L3 = (B2*D1 - B1*D2)/den; L4 = (B2*C1 - B1*C2)/den
    u = 1 + L2**2 + L4**2; c0 = C[:, 0]
    v = 2*(L3*L4 + L1*L2 - c0[:, 0]*L4 - c0[:, 1]*L2 - c0[:, 2])
    w = c0[:, 0]**2 + c0[:, 1]**2 + c0[:, 2]**2 + L3**2 + L1**2 - 2*c0[:, 0]*L3 - 2*c0[:, 1]*L1 - L_B**2
    z = (-v + np.sqrt(v**2 - 4*u*w)) / (2*u)
    return np.column_stack([L3 + L4*z, L1 + L2*z, z])
def fk(q): return fk_vec(q[None, :])[0]

def regressor_batch(q, a, P):
    n = len(q); s, c = np.sin(q), np.cos(q); x, y, z = P[:, 0], P[:, 1], P[:, 2]
    Y = np.zeros((n, 3, 14)); Y[:, :, 0] = a; Y[:, :, 4] = c
    for i, (cx, cy, cz, cs) in enumerate([(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]):
        Y[:, i, cx] = -x*s[:, i]; Y[:, i, cy] = -y*s[:, i]; Y[:, i, cz] = z*c[:, i]; Y[:, i, cs] = s[:, i]
    return Y
def regressor(q, a, p): return regressor_batch(q[None, :], np.asarray(a, float)[None, :], np.asarray(p)[None, :])[0]

def caputo_l1(x, h, alpha):
    x = np.asarray(x, float); N = len(x); k = np.arange(N); b = (k + 1.0)**(1 - alpha) - k**(1.0 - alpha)
    d = np.vstack([np.zeros((1, x.shape[1])), np.diff(x, axis=0)])
    return fftconvolve(d, b[:, None], axes=0)[:N] * h**(-alpha) / Gamma(2 - alpha)

## Constrained multibody model of the delta robot

x = (q, p) ∈ R⁶, constraints Φ_i = |p − C_i(q_i)|² − L_B² = 0, with C_i the elbow centre (shifted by the platform radius, as in the forward kinematics). Kinetic energy: arms ½ I_A q̇², elbow masses ½ m_E |Ċ_i|² (= ½ m_E L_A² q̇²), each forearm parallelogram a pair of slender rods of total mass m_F with end velocities Ċ_i and ṗ, T = m_F/6 (|Ċ_i|² + Ċ_i·ṗ + |ṗ|²) (exact for uniform rods; their spin about the own axis carries no energy), platform ½ (m_P + m) |ṗ|². Gravity along +z. Equations of motion

M(q) ẍ + Φ_xᵀ λ = Q(q, q̇) + [τ − τ_f − τ_mem; 0],  Φ_x ẍ = −γ − 2ω Φ̇ − ω² Φ (Baumgarte),

solved as one 9 × 9 linear system per step, symplectic Euler at 0.5 ms. The inverse dynamics of a prescribed joint motion uses the same equations with the constraints solved exactly (p by forward kinematics, ṗ and p̈ from Φ_x).

In [ ]:
ANG = 2*np.pi*np.arange(3)/3; CA, SA = np.cos(ANG), np.sin(ANG); EZ = np.array([0, 0, 1.0])
def eff(p=MBP):
    """Effective arm inertia (arm + elbow + rod end), effective arm gravity moment, rod and platform mass."""
    return (p["I_A"] + p["m_E"]*L_A**2 + p["m_F"]*L_A**2/3, p["mr_A"] + p["m_E"]*L_A + p["m_F"]*L_A/2, p["m_F"], p["m_P"])
def elbows(q):
    """C_i(q_i) and its first and second derivatives, (n, 3 chains, 3 coordinates)."""
    rd = R_BASE - R_PLAT; s, c = np.sin(q), np.cos(q)
    C = np.stack([(rd + L_A*c)*CA, (rd + L_A*c)*SA, L_A*s], -1)
    C1 = np.stack([-L_A*s*CA, -L_A*s*SA, L_A*c], -1)
    C2 = np.stack([-L_A*c*CA, -L_A*c*SA, -L_A*s], -1)
    return C, C1, C2

def mb_inverse(q, qd, qdd, m, p=MBP):
    """Rigid multibody inverse dynamics of a prescribed joint motion (batched). Returns joint torques (n, 3) and multipliers."""
    Ie, Ge, mF, mP = eff(p); mt = mP + np.broadcast_to(np.asarray(m, float), (len(q),))
    P = fk_vec(q); C, C1, C2 = elbows(q); d = P[:, None, :] - C
    Pp = 2*d; Pq = -2*np.einsum("nik,nik->ni", d, C1)
    pd = np.linalg.solve(Pp, -(Pq*qd)[..., None])[..., 0]
    w = pd[:, None, :] - C1*qd[..., None]
    gam = 2*np.einsum("nik,nik->ni", w, w) - 2*np.einsum("nik,nik->ni", d, C2)*qd**2
    pdd = np.linalg.solve(Pp, (-Pq*qdd - gam)[..., None])[..., 0]
    r = ((mt + mF)[:, None]*pdd + mF/6*np.einsum("nik,ni->nk", C1, qdd) + mF/6*np.einsum("nik,ni->nk", C2, qd**2)
         - G_ACC*(mt + 1.5*mF)[:, None]*EZ)
    lam = np.linalg.solve(np.transpose(Pp, (0, 2, 1)), -r[..., None])[..., 0]
    return Ie*qdd + mF/6*np.einsum("nik,nk->ni", C1, pdd) - G_ACC*Ge*np.cos(q) + Pq*lam, lam

class MBPlant:
    """Forward dynamics of the constrained multibody delta (Lagrange multipliers, Baumgarte, symplectic Euler)."""
    def __init__(self, m, p=MBP, wb=BAUMGARTE_W):
        self.Ie, self.Ge, self.mF, self.mP = eff(p); self.m = m; self.wb = wb
    def reset(self, q0, qd0):
        self.q = np.array(q0, float); self.qd = np.array(qd0, float); self.p = fk(self.q)
        C, C1, _ = (a[0] for a in elbows(self.q[None])); d = self.p - C
        self.pd = np.linalg.solve(2*d, 2*np.einsum("ik,ik->i", d, C1)*self.qd)
    def step(self, tau, h):
        q, qd, p, pd, mF = self.q, self.qd, self.p, self.pd, self.mF; mt = self.mP + self.m
        C, C1, C2 = (a[0] for a in elbows(q[None])); d = p - C
        Pq = -2*np.einsum("ik,ik->i", d, C1); Pp = 2*d
        M = np.zeros((9, 9)); M[:3, :3] = self.Ie*np.eye(3); M[3:6, 3:6] = (mt + mF)*np.eye(3)
        M[:3, 3:6] = mF/6*C1; M[3:6, :3] = mF/6*C1.T
        M[6:, :3] = np.diag(Pq); M[6:, 3:6] = Pp; M[:3, 6:] = np.diag(Pq); M[3:6, 6:] = Pp.T
        w = pd - C1*qd[:, None]
        gam = 2*np.einsum("ik,ik->i", w, w) - 2*np.einsum("ik,ik->i", d, C2)*qd**2
        phi = np.einsum("ik,ik->i", d, d) - L_B**2; phid = Pq*qd + Pp @ pd
        rhs = np.concatenate([tau + G_ACC*self.Ge*np.cos(q), G_ACC*(mt + 1.5*mF)*EZ - mF/6*(C2.T @ qd**2),
                              -gam - 2*self.wb*phid - self.wb**2*phi])
        s = np.linalg.solve(M, rhs)
        self.qd = qd + h*s[:3]; self.pd = pd + h*s[3:6]; self.q = q + h*self.qd; self.p = p + h*self.pd
        return phi
    def energy(self):
        C, C1, _ = (a[0] for a in elbows(self.q[None]))
        T = (0.5*self.Ie*self.qd @ self.qd + 0.5*(self.mP + self.m + self.mF)*self.pd @ self.pd
             + self.mF/6*np.sum((C1*self.qd[:, None]) @ self.pd))
        return T - G_ACC*(self.Ge*np.sum(np.sin(self.q)) + (self.mP + self.m + 1.5*self.mF)*self.p[2])

## Joint dissipation: viscous + Coulomb + Caputo (scenario caputo) and LuGre (scenario lugre)

LuGre per joint: ż = v − σ₀|v| z / g(v), g(v) = F_c + (F_s − F_c) e^{−(v/v_s)²}, τ = σ₀ z + σ₁ e^{−(v/v_s)²} ż + σ₂ v. The bristle state is advanced with the exact solution for a velocity held over the step (stable for the stiff presliding regime). Its **memory part** is τ minus the steady-state curve g(v) sgn v + σ₂ v.

In [ ]:
def friction_cv(qd):
    f = FRICTION; return f["b_v"]*qd + f["f_c"]*np.tanh(qd/f["eps"])

class LuGre:
    def __init__(self, n=3): self.z = np.zeros(n)
    def step(self, v, h):
        L = LUGRE; e = np.exp(-(v/L["vs"])**2); g = L["Fc"] + (L["Fs"] - L["Fc"])*e
        zss = np.sign(v)*g/L["s0"]; z_new = zss + (self.z - zss)*np.exp(-L["s0"]*np.abs(v)/g*h)
        zd = (z_new - self.z)/h; self.z = z_new
        tau = L["s0"]*self.z + L["s1"]*e*zd + L["s2"]*v
        return tau, tau - (np.sign(v)*g + L["s2"]*v)

def lugre_along(qd, h):
    lg = LuGre(); tau = np.zeros_like(qd); mem = np.zeros_like(qd)
    for n in range(len(qd)): tau[n], mem[n] = lg.step(qd[n], h)
    return tau, mem

## Checks of the multibody model (saved to `NB08_fPINN_Verificacion.csv`)

1. Free motion (τ = 0, no dissipation) for 2 s from rest: energy drift and constraint residual.
2. Replay: torques from the inverse dynamics of a random trajectory, applied open loop to the forward dynamics, must reproduce the joint motion.

In [ ]:
def random_trajectory_full(rng, n_wp=6, v_range=(0.03, 0.8), t_min=0.3):     # RNG use identical to NB02 / NB04
    W = np.column_stack([rng.uniform(-0.12, 0.12, n_wp), rng.uniform(-0.12, 0.12, n_wp), rng.uniform(0.49, 0.56, n_wp)])
    dist = np.linalg.norm(np.diff(W, axis=0), axis=1)
    v = np.exp(rng.uniform(np.log(v_range[0]), np.log(v_range[1]), n_wp - 1))
    dur = np.maximum(dist / v, t_min); m_load = float(rng.integers(0, 7))*PAYLOAD_SCALE
    edges = np.concatenate([[0.0], np.cumsum(dur)]); t_g = np.arange(0.0, edges[-1], dt)
    seg = np.clip(np.searchsorted(edges, t_g, side="right") - 1, 0, n_wp - 2)
    ts = (t_g - edges[seg]) / dur[seg]; s = 6*ts**5 - 15*ts**4 + 10*ts**3
    P = W[seg] + (W[seg + 1] - W[seg])*s[:, None]
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(q=q, qd=qd, qdd=qdd, P=P, m=m_load)

ver = []
pl = MBPlant(0.5); pl.reset(ik_vec(np.array([[0.05, -0.03, 0.52]]))[0], np.zeros(3)); E0 = pl.energy(); Es, Ph = [], []
for i in range(int(2.0/dt)): Ph.append(np.abs(pl.step(np.zeros(3), dt)).max()); Es.append(pl.energy())
p_err = np.linalg.norm(pl.p - fk(pl.q))
ver += [dict(Check="free motion 2 s: max energy drift (J)", Value=np.max(np.abs(np.array(Es) - E0))),
        dict(Check="free motion 2 s: initial potential energy magnitude (J)", Value=abs(E0)),
        dict(Check="free motion 2 s: max constraint residual |Phi| (m^2)", Value=max(Ph)),
        dict(Check="free motion 2 s: final |p - FK(q)| (m)", Value=p_err)]
T = random_trajectory_full(np.random.default_rng(3)); tau_id, _ = mb_inverse(T["q"], T["qd"], T["qdd"], T["m"])
pl = MBPlant(T["m"]); pl.reset(T["q"][0], T["qd"][0]); err, Ph = [], []
for i in range(len(T["q"]) - 1):
    Ph.append(np.abs(pl.step(tau_id[i], dt)).max()); err.append(np.abs(pl.q - T["q"][i + 1]).max())
ver += [dict(Check="replay of inverse dynamics: max |q error| (rad)", Value=max(err)),
        dict(Check="replay of inverse dynamics: duration (s)", Value=len(T["q"])*dt),
        dict(Check="replay of inverse dynamics: max |Phi| (m^2)", Value=max(Ph))]
ver = pd.DataFrame(ver); ver.to_csv("NB08_fPINN_Verificacion" + SUF + ".csv", index=False); print(ver.to_string(index=False))

## Reduced nominal model identified on the multibody plant (`NB08_fPINN_ModeloReducido.csv`)

θ₀ (14 parameters) and m₀ of γ(m) = (m₀ + m)/m₀ by least squares on rigid multibody torques of `N_CAL_TRAJ` separate trajectories (all payloads). The residual is the structural gap of the reduced model.

In [ ]:
cal = [random_trajectory_full(np.random.default_rng(CAL_SEED + k)) for k in range(N_CAL_TRAJ)]
cal = [(T["q"][::10], T["qdd"][::10], T["P"][::10], T["m"], mb_inverse(T["q"][::10], T["qd"][::10], T["qdd"][::10], T["m"])[0]) for T in cal]
y_cal = np.concatenate([t.reshape(-1) for *_, t in cal]); best = None
for m0 in np.round(np.arange(0.25, 6.01, 0.25), 2):
    Y = np.vstack([((m0 + m)/m0)*regressor_batch(q, a, P).reshape(-1, 14) for q, a, P, m, _ in cal])
    th = np.linalg.lstsq(Y, y_cal, rcond=None)[0]; r = np.sqrt(np.mean((Y @ th - y_cal)**2))
    if best is None or r < best[0]: best = (r, m0, th)
GAP_RMS, M_BASE, THETA_NOM = best
def gamma_payload(m): return (M_BASE + np.asarray(m, float)) / M_BASE
red = pd.DataFrame([dict(Parameter=f"theta_{j+1}", Value=v) for j, v in enumerate(THETA_NOM)]
                   + [dict(Parameter="m0 (kg)", Value=M_BASE), dict(Parameter="structural gap RMS (N m)", Value=GAP_RMS),
                      dict(Parameter="rigid torque RMS (N m)", Value=np.sqrt(np.mean(y_cal**2)))]
                   + [dict(Parameter=f"MBP {k}", Value=v) for k, v in MBP.items()]
                   + [dict(Parameter=n, Value=v) for n, v in zip(("I_eff (kg m2)", "G_eff (kg m)"), eff()[:2])])
red.to_csv("NB08_fPINN_ModeloReducido" + SUF + ".csv", index=False)
print(f"reduced model on the multibody plant: m0 = {M_BASE} kg | structural gap {1e3*GAP_RMS:.0f} mN m RMS "
      f"of {1e3*np.sqrt(np.mean(y_cal**2)):.0f} mN m | theta1 = {THETA_NOM[0]:.4f}")
del cal, y_cal

## Data: NB02/NB04 trajectories, torques of the multibody plant in both scenarios

Inputs (q, q̇, q̈, m, p) and the logged history behind the L1 features are those of NB04 (clean). Labels: rigid multibody torque + scenario dissipation. `base` is the reduced nominal model (for reference). The true memory torque is stored to compare with the identified one.

In [ ]:
def log_and_label(T, margin_s=0.05):
    q, qd, qdd, P, m = T["q"], T["qd"], T["qdd"], T["P"], T["m"]
    rig, _ = mb_inverse(q, qd, qdd, m)
    frac = B_FRAC*caputo_l1(q, dt, ALPHA_MAIN); lug, lug_mem = lugre_along(qd, dt)
    nom = gamma_payload(m)*np.einsum("nij,j->ni", regressor_batch(q, qdd, P), THETA_NOM)
    mg = int(margin_s/dt); idx = np.arange(0, len(q), SUBSAMPLE); keep = (idx >= mg) & (idx < len(q) - mg)
    h = dt*SUBSAMPLE; G = np.stack([caputo_l1(q[idx], h, a)[keep] for a in ALPHA_GRID], axis=1).astype(np.float32)
    i = idx[keep]
    return dict(X=np.column_stack([q[i], qd[i], qdd[i], np.full(len(i), m), P[i]]), nom=nom[i], G=G,
                y_caputo=(rig + friction_cv(qd) + frac)[i], mem_caputo=frac[i],
                y_lugre=(rig + lug)[i], mem_lugre=lug_mem[i])

def build(n_traj, rng):
    parts = [log_and_label(random_trajectory_full(rng)) for _ in range(n_traj)]
    D = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    D["traj"] = np.concatenate([np.full(len(p["X"]), j) for j, p in enumerate(parts)]); return D

t0 = time.time(); rng = np.random.default_rng(DATA_SEED)
DATA = (build(N_TRAIN_MAX, rng), build(N_VAL_TRAJ, rng), build(N_TEST_TRAJ, rng)); T_DATA = time.time() - t0
Dte = DATA[2]
print(f"data in {T_DATA:.0f} s | train {len(DATA[0]['X']):,} samples | test RMS: rigid+dissipation caputo {1e3*np.sqrt(np.mean(Dte['y_caputo']**2)):.0f} mN m, "
      f"Caputo torque {1e3*np.sqrt(np.mean(Dte['mem_caputo']**2)):.1f} mN m, LuGre memory part {1e3*np.sqrt(np.mean(Dte['mem_lugre']**2)):.1f} mN m, "
      f"nominal error caputo {1e3*np.sqrt(np.mean((Dte['y_caputo']-Dte['nom'])**2)):.0f} mN m, lugre {1e3*np.sqrt(np.mean((Dte['y_lugre']-Dte['nom'])**2)):.0f} mN m")

def labels(D, scn): return dict(X=D["X"], y=D[f"y_{scn}"], base=D["nom"], G=D["G"], traj=D["traj"], mem=D[f"mem_{scn}"])
def subset(D, size):
    if size >= int(D["traj"].max()) + 1: return D
    keep = D["traj"] < size; return {k: v[keep] for k, v in D.items()}
def ram_gb():
    try:
        import psutil; return psutil.Process().memory_info().rss/1e9
    except Exception: return float("nan")

## Training: PINN and fPINN of NB02/NB04 (fast batches), output bias at the reduced model identified above

In [ ]:
GA = torch.tensor(ALPHA_GRID, dtype=torch.float32); DA = float(ALPHA_GRID[1] - ALPHA_GRID[0])
def interp_D(G, alpha):
    u = torch.clamp((alpha - GA[0]) / DA, 0.0, len(ALPHA_GRID) - 1.000001)
    i0 = torch.floor(u).long(); w = u - i0
    return (1 - w)*G[:, i0, :] + w*G[:, i0 + 1, :]

def epoch_batches(n, gen, bs):
    torch.empty((), dtype=torch.int64).random_(generator=gen)
    perm = torch.randperm(n, generator=gen); torch.randperm(n, generator=gen)
    return perm.split(bs)

def train_ff_model(kind, d_tr, d_va, seed, cfg=TRAIN_CFG):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    x_mean, x_std = d_tr["X"].mean(0), d_tr["X"].std(0) + 1e-8
    y_mean, y_std = d_tr["y"].mean(0), d_tr["y"].std(0)
    PHYS_IDX = [0, 1, 2, 6, 7, 8, 10, 11, 12]; frac = kind == "fPINN"

    class FFNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(13, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 14))
            with torch.no_grad():
                self.net[-1].weight.mul_(0.1); self.net[-1].bias.copy_(torch.tensor(THETA_NOM, dtype=torch.float32))
            if frac:
                a0 = (ALPHA_INIT - 0.05)/0.925
                self.a_raw = nn.Parameter(torch.tensor(np.log(a0/(1 - a0)), dtype=torch.float32))
                self.B = nn.Parameter(torch.zeros(3))
        def alpha(self): return 0.05 + 0.925*torch.sigmoid(self.a_raw)
        def forward(self, xs, xp, G):
            th = self.net(xs); q, qdd = xp[:, 0:3], xp[:, 3:6]; x, y, z = xp[:, 6], xp[:, 7], xp[:, 8]
            cols = [(1, 2, 3, 5), (6, 7, 8, 9), (10, 11, 12, 13)]
            tau = torch.stack([th[:, 0]*qdd[:, i] - th[:, a]*x*torch.sin(q[:, i]) - th[:, b]*y*torch.sin(q[:, i])
                               + th[:, c]*z*torch.cos(q[:, i]) + th[:, 4]*torch.cos(q[:, i]) + th[:, d]*torch.sin(q[:, i])
                               for i, (a, b, c, d) in enumerate(cols)], dim=1)
            return tau + self.memory_torque(G) if frac else tau
        def memory_torque(self, G):
            return self.B*interp_D(G, self.alpha()) if frac else torch.zeros(len(G), 3)

    def to_tensors(d):
        xs = torch.tensor((d["X"] - x_mean)/x_std, dtype=torch.float32)
        xp = torch.tensor(d["X"][:, PHYS_IDX], dtype=torch.float32)
        G = torch.from_numpy(d["G"]) if frac else torch.zeros(len(xs), 1, 1)
        return TensorDataset(xs, xp, G, torch.tensor(d["y"], dtype=torch.float32))

    T_tr, T_va = to_tensors(d_tr).tensors, to_tensors(d_va).tensors
    gen = torch.Generator().manual_seed(seed); n_tr = len(T_tr[0])
    model = FFNet()
    groups = [{"params": model.net.parameters(), "weight_decay": cfg["weight_decay"]}]
    if frac: groups.append({"params": [model.a_raw, model.B], "weight_decay": 0.0})
    opt = torch.optim.Adam(groups, lr=cfg["lr"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=cfg["lr_patience"])
    loss_fn = nn.MSELoss(); best, best_state, wait = np.inf, None, 0; t_tr = time.time()
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        for idx in epoch_batches(n_tr, gen, cfg["batch_size"]):
            xs, xp, G, y = [t[idx] for t in T_tr]
            opt.zero_grad(); loss = loss_fn(model(xs, xp, G), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vs, vn = 0.0, 0
        with torch.no_grad():
            for k0 in range(0, len(T_va[0]), cfg["batch_size"]):
                xs, xp, G, y = [t[k0:k0 + cfg["batch_size"]] for t in T_va]
                vs += loss_fn(model(xs, xp, G), y).item()*len(y); vn += len(y)
        val = vs/vn; sched.step(val)
        if val < best - 1e-12: best, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
        else: wait += 1
        if epoch % cfg["log_every"] == 0 or epoch == 1 or wait >= cfg["patience"]:
            ex = f" | α {model.alpha().item():.3f} B {np.round(model.B.detach().numpy(), 4)}" if frac else ""
            print(f"    [{kind} {d_tr['traj'].max()+1}traj seed{seed}] epoch {epoch:3d} | val MSE {val:.3e} | best {best:.3e}{ex} | {time.time()-t_tr:.0f} s", flush=True)
        if wait >= cfg["patience"]: break
    model.load_state_dict(best_state); model.eval()
    info = dict(epochs=epoch, best_val_mse=best, train_s=time.time() - t_tr)
    lins = [m_ for m_ in model.net if isinstance(m_, nn.Linear)]
    info["export"] = dict(kind=kind, **{k: t_.detach().numpy().astype(float) for k, t_ in
                          zip(("W1", "b1", "W2", "b2", "W3", "b3"), [t for l_ in lins for t in (l_.weight, l_.bias)])},
                          x_mean=x_mean, x_std=x_std, alpha=model.alpha().item() if frac else 0.0,
                          B=model.B.detach().numpy().astype(float) if frac else np.zeros(3))
    if frac: info.update(alpha_hat=model.alpha().item(), B_hat=model.B.detach().numpy().astype(float))
    return model, to_tensors, info

def evaluate(model, to_tensors, d):
    """Torque RMSE, identified and true memory torque RMS, energy dissipated by the identified and true memory torque."""
    xs, xp, G, y = to_tensors(d).tensors
    with torch.no_grad(): pred = model(xs, xp, G).numpy(); mh = model.memory_torque(G).numpy()
    qd, h = d["X"][:, 3:6], dt*SUBSAMPLE
    return dict(rmse=float(np.sqrt(np.mean((pred - d["y"])**2))), mem_hat=float(np.sqrt(np.mean(mh**2))),
                mem_true=float(np.sqrt(np.mean(d["mem"]**2))), E_hat=float(h*np.sum(mh*qd)), E_true=float(h*np.sum(d["mem"]*qd)))

## Adept-like references (NB03/NB04) for the open-loop torque check and the closed loop

In [ ]:
P_READY = np.array([0.0, 0.0, 0.5])
WAYPOINTS = [P_READY, np.array([.10, .10, .55]), np.array([.10, -.10, .55]),
             np.array([-.10, -.10, .55]), np.array([-.10, .10, .55]), np.array([.10, .10, .55]), P_READY]
BASE_T = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.50]); H_P = 0.05
def adept_full(scale, m=0.0):
    nt = BASE_T*scale; n = int(round(nt[-1]*FS)) + 1; t = np.arange(n)*dt
    seg = np.clip(np.searchsorted(nt[1:], t, side="right"), 0, len(WAYPOINTS) - 2)
    tau = np.clip((t - nt[seg]) / (nt[seg+1] - nt[seg]), 0, 1); s = 6*tau**5 - 15*tau**4 + 10*tau**3
    W = np.array(WAYPOINTS); P = W[seg] + (W[seg+1] - W[seg])*s[:, None]; P[:, 2] -= H_P*np.sin(np.pi*tau)**2
    q = ik_vec(P); qd = np.gradient(q, dt, axis=0, edge_order=2); qdd = np.gradient(qd, dt, axis=0, edge_order=2)
    return dict(t=t, q=q, qd=qd, qdd=qdd, P=P, m=m, N=n)
REFS_OL = {}
for sc in (6.4, 3.2, 1.6):
    for m in (0.0, 1.0):
        R = log_and_label(adept_full(sc, m), margin_s=0.0); R["traj"] = np.zeros(len(R["X"]), int); REFS_OL[(sc, m)] = R

## Train every network (caputo first; cached and resumable)

In [ ]:
LOG_CSV = "NB08_fPINN_TrainingLog" + SUF + ".csv"
def tag(scn, kind, size, seed): return f"{scn}_{kind}_{size}traj_seed{seed}_e{E}"
rows, sec_epoch = [], []; t_all = time.time()
def eta(j):
    left = [x for x in EXPERIMENTS[j:] if not os.path.exists(os.path.join(CACHE_DIR, tag(*x) + ".pkl"))]
    return len(left)*110*(np.median(sec_epoch) if sec_epoch else 1.6)
print(f"estimated training time on this PC before measuring: {eta(0)/3600:.1f} h (+ closed loop, see below)")
for j, (scn, kind, size, seed) in enumerate(EXPERIMENTS, 1):
    name = tag(scn, kind, size, seed); f_c = os.path.join(CACHE_DIR, name + ".pkl"); f_w = os.path.join(WEIGHTS_OUT, f"NB08_fPINN_{name}.npz")
    if os.path.exists(f_c) and os.path.exists(f_w):
        rows.append(pickle.load(open(f_c, "rb"))); print(f"[{j}/{len(EXPERIMENTS)}] {name}: loaded from cache"); continue
    TR, VA, TE = labels(subset(DATA[0], size), scn), labels(DATA[1], scn), labels(DATA[2], scn)
    model, tt, info = train_ff_model(kind, TR, VA, seed)
    ev = evaluate(model, tt, TE)
    row = dict(Scenario=scn, Kind=kind, Trajectories=size, Seed=seed, Epochs=info["epochs"],
               **{"Train time (s)": info["train_s"], "Val RMSE (N m)": evaluate(model, tt, VA)["rmse"], "Test RMSE (N m)": ev["rmse"],
                  "Nominal test RMSE (N m)": float(np.sqrt(np.mean((TE["y"] - TE["base"])**2))),
                  "Memory torque RMS (N m)": ev["mem_hat"], "True memory torque RMS (N m)": ev["mem_true"],
                  "Memory energy identified (J)": ev["E_hat"], "Memory energy true (J)": ev["E_true"]})
    for sc in (6.4, 3.2, 1.6):
        for m in (0.0, 1.0): row[f"Adept {sc} {m:.0f} kg (N m)"] = evaluate(model, tt, labels(REFS_OL[(sc, m)], scn))["rmse"]
    if "alpha_hat" in info:
        row.update({"alpha hat": info["alpha_hat"], "B hat mean": float(info["B_hat"].mean()),
                    "B hat": [round(float(b), 4) for b in info["B_hat"]], "Detected (B hat > %.3f)" % DETECT_B: bool(info["B_hat"].mean() > DETECT_B)})
    np.savez(f_w, **info["export"]); pickle.dump(row, open(f_c, "wb"))
    rows.append(row); pd.DataFrame(rows).to_csv(LOG_CSV, index=False); sec_epoch.append(info["train_s"]/info["epochs"])
    print(f"[{j}/{len(EXPERIMENTS)}] {name}: test {ev['rmse']*1e3:.2f} mN m (nominal {row['Nominal test RMSE (N m)']*1e3:.0f})"
          + (f" | α̂ {row['alpha hat']:.3f} B̂ {row['B hat']}" if "alpha hat" in row else "")
          + f" | memory {ev['mem_hat']*1e3:.2f} vs true {ev['mem_true']*1e3:.2f} mN m | {info['train_s']/info['epochs']:.2f} s/epoch"
          + f" | {(time.time()-t_all)/60:.0f} min | training left ≈ {eta(j)/3600:.1f} h | RAM {ram_gb():.2f} GB", flush=True)
    del model, tt, info, TR, VA, TE; gc.collect()
df = pd.DataFrame(rows); df.to_csv(LOG_CSV, index=False)
del DATA, Dte; _ = gc.collect()

## Closed loop: CTC + I of NB03 on the multibody plant

Same gains, 5 N m limit and 2 ms memory update as NB03/NB04. The fPINN adds B̂ D^α̂ q from the measured history (full-memory L1). The oracle uses the exact multibody inverse dynamics plus the true dissipation: the Caputo term from the measured history, or a LuGre observer driven by the measured velocity. The Cartesian error is measured on the platform position of the multibody plant.

In [ ]:
KP_PD, KD_PD, KI_RATIO = 150.0, 10.0, 10.0
KP_EQ, KD_EQ = KP_PD/0.070, KD_PD/0.070             # NB03 gains (θ1 of the paper's reduced model)
T_C = 0.002; TAU_MAX = 5.0

def silu(z): return z / (1.0 + np.exp(-z))
class NumpyFF:
    def __init__(self, path):
        d = np.load(path, allow_pickle=False); self.kind = str(d["kind"])
        self.W = [d[k].astype(float) for k in ("W1", "b1", "W2", "b2", "W3", "b3")]
        self.x_mean, self.x_std = d["x_mean"].astype(float), d["x_std"].astype(float)
        self.alpha, self.B = float(d["alpha"]), d["B"].astype(float)
    def F(self, q, qd, qdd, m):
        p = fk(q); xs = (np.concatenate([q, qd, qdd, [m], p]) - self.x_mean) / self.x_std
        W1, b1, W2, b2, W3, b3 = self.W
        return regressor(q, qdd, p) @ (W3 @ silu(W2 @ silu(W1 @ xs + b1) + b2) + b3)

class FracHistory:
    """Online L1 Caputo derivative (full memory); returns (1, 3)."""
    def __init__(self, alpha, h, n_max):
        k = np.arange(n_max + 1); self.b = ((k + 1.0)**(1 - alpha) - k**(1.0 - alpha))[::-1].copy()
        self.c = h**(-alpha)/Gamma(2 - alpha); self.dq = np.zeros((n_max + 1, 3)); self.n = 0; self.q_last = None
    def push(self, q):
        if self.q_last is None: self.q_last = q.copy(); return np.zeros(3)
        self.n += 1; self.dq[self.n] = q - self.q_last; self.q_last = q.copy()
        return self.c*(self.b[-self.n:] @ self.dq[1:self.n + 1])

def F_nominal(q, qd, qdd, m): return gamma_payload(m)*(regressor(q, qdd, fk(q)) @ THETA_NOM)
def F_rigid(q, qd, qdd, m): return mb_inverse(q[None], qd[None], qdd[None], m)[0][0]
class OracleLuGre:
    def __init__(self): self.lg = LuGre()
    def __call__(self, q, qd, qdd, m): return F_rigid(q, qd, qdd, m) + self.lg.step(qd, dt)[0]

class Controller:
    """CTC + I (NB03) + Caputo memory term updated every T_C from the measured angles."""
    def __init__(self, F, alpha=0.5, B=np.zeros(3)):
        self.F, self.alpha, self.B = F, alpha, np.asarray(B, float); self.step = int(round(T_C/dt))
    def reset(self, N):
        self.I = np.zeros(3); self.mem = np.zeros(3)
        self.H = FracHistory(self.alpha, T_C, N//self.step + 2) if self.B.any() else None
    def __call__(self, R, i, q, qd, m):
        if self.H is not None and i % self.step == 0: self.mem = self.B*self.H.push(q)
        e = R["q"][i] - q; ed = R["qd"][i] - qd; self.I += e*dt
        v = R["qdd"][i] + KD_EQ*ed + KP_EQ*e + KI_RATIO*KP_EQ*self.I
        return self.F(q, qd, v, m) + self.mem

def simulate(ctrl, R, m, scn):
    ctrl.reset(R["N"]); pl = MBPlant(m); pl.reset(R["q"][0], R["qd"][0])
    Hp = FracHistory(ALPHA_MAIN, dt, R["N"]) if scn == "caputo" else None; lg = LuGre()
    P = np.zeros((R["N"], 3)); T = np.zeros((R["N"], 3)); ph = 0.0
    for i in range(R["N"]):
        q, qd = pl.q.copy(), pl.qd.copy(); P[i] = pl.p
        tau = np.clip(ctrl(R, i, q, qd, m), -TAU_MAX, TAU_MAX); T[i] = tau
        diss = friction_cv(qd) + B_FRAC*Hp.push(q) if scn == "caputo" else lg.step(qd, dt)[0]
        ph = max(ph, np.abs(pl.step(tau - diss, dt)).max())
        if not np.all(np.isfinite(pl.q)) or np.abs(pl.q).max() > 3: return None
    e = 1000*np.linalg.norm(R["P"] - P, axis=1)
    return {"Cartesian RMSE (mm)": np.sqrt(np.mean(e**2)), "Max error (mm)": e.max(), "RMS torque (N m)": np.sqrt(np.mean(T**2)),
            "TV (N m)": np.sum(np.linalg.norm(np.diff(T, axis=0), axis=1)), "Peak |tau| (N m)": np.abs(T).max(),
            "Saturated (%)": 100*np.mean(np.abs(T) >= TAU_MAX - 1e-9), "Max |Phi| (m^2)": ph}

REFS_CL = {sc: adept_full(sc) for sc in SPEED_SCALES}
def models_for_cl():
    out = []
    for scn in SCENARIOS:
        out.append((scn, "nominal", 0, 0, lambda: Controller(F_nominal)))
        out.append((scn, "oracle", 0, 0, (lambda: Controller(F_rigid_cv, ALPHA_MAIN, np.full(3, B_FRAC))) if scn == "caputo"
                                          else (lambda: Controller(OracleLuGre()))))
    for scn, kind, size, seed in EXPERIMENTS:
        f = os.path.join(WEIGHTS_OUT, f"NB08_fPINN_{tag(scn, kind, size, seed)}.npz")
        if not os.path.exists(f): print("missing weights:", tag(scn, kind, size, seed)); continue
        def make(f=f):
            n = NumpyFF(f); return Controller(n.F, n.alpha, n.B)
        out.append((scn, kind, size, seed, make))
    return out
def F_rigid_cv(q, qd, qdd, m): return F_rigid(q, qd, qdd, m) + friction_cv(qd)

CL_CSV = "NB08_fPINN_ClosedLoop" + SUF + ".csv"
key = lambda d: (d["Scenario"], d["Model"], int(d["Data (traj.)"]), int(d["Seed"]), float(d["Speed scale"]), float(d["Payload (kg)"]))
cl_rows = pd.read_csv(CL_CSV).to_dict("records") if os.path.exists(CL_CSV) else []
done = {key(r) for r in cl_rows}
jobs = [(M, sc, m) for M in models_for_cl() for sc in SPEED_SCALES for m in PAYLOADS]
todo = [(M, sc, m) for M, sc, m in jobs if key({"Scenario": M[0], "Model": M[1], "Data (traj.)": M[2], "Seed": M[3],
                                                 "Speed scale": sc, "Payload (kg)": m}) not in done]
print(f"closed loop: {len(todo)} runs to do"); t_cl = time.time()
for j, ((scn, label, size, seed, make), sc, m) in enumerate(todo, 1):
    ctrl = make(); t0 = time.time(); res = simulate(ctrl, REFS_CL[sc], m, scn)
    cl_rows.append(dict(Scenario=scn, Controller="CTC + I", Model=label, **{"Data (traj.)": size}, Seed=seed,
                        **{"Speed scale": sc, "Payload (kg)": m, "alpha hat": ctrl.alpha if ctrl.B.any() else np.nan,
                           "B hat mean": ctrl.B.mean(), "Run time (s)": time.time() - t0}, Unstable=res is None, **(res or {})))
    pd.DataFrame(cl_rows).to_csv(CL_CSV, index=False)
    print(f"[{j}/{len(todo)}] {scn} | {sc} s {m:.0f} kg | {label:8s} {size:3d} traj s{seed} | "
          + ("UNSTABLE" if res is None else f"RMSE {1e3*res['Cartesian RMSE (mm)']:.1f} µm, peak {res['Peak |tau| (N m)']:.2f} N m")
          + f" | {time.time()-t0:.0f} s | {(time.time()-t_cl)/60:.0f} min", flush=True)
cl = pd.DataFrame(cl_rows)

## Summary (`NB08_fPINN_Summary.csv`): fPINN − PINN paired over seeds, identified memory, dissipated energy

In [ ]:
from scipy import stats
pd.set_option("display.width", 250); out = []
def paired(a, b, name):
    s = a.index.intersection(b.index); d = (a[s] - b[s]).values
    if len(d) < 2: return dict(Comparison=name, n=len(d), **{"mean diff": d.mean() if len(d) else np.nan})
    ci = stats.t.ppf(0.975, len(d) - 1)*d.std(ddof=1)/np.sqrt(len(d))
    return dict(Comparison=name, n=len(d), **{"mean diff": d.mean(), "rel diff (%)": 100*d.mean()/b[s].mean(),
                                              "CI95 half": ci, "p (paired t)": stats.ttest_rel(a[s], b[s]).pvalue})
cols = ["Test RMSE (N m)", "Adept 1.6 0 kg (N m)", "Adept 1.6 1 kg (N m)"]
print("Torque RMSE (mN m), mean ± std over seeds:")
print((df.groupby(["Scenario", "Kind"])[cols + ["Nominal test RMSE (N m)"]].agg(["mean", "std"])*1e3).round(2).to_string())
fr = df[df.Kind == "fPINN"]
if len(fr):
    print("\nfPINN identified memory (mean ± std over seeds):")
    print(fr.groupby("Scenario")[["alpha hat", "B hat mean", "Memory torque RMS (N m)", "True memory torque RMS (N m)",
                                  "Memory energy identified (J)", "Memory energy true (J)"]].agg(["mean", "std"]).round(4).to_string())
for scn in SCENARIOS:
    g = df[df.Scenario == scn]
    for c in cols:
        out.append(dict(Scenario=scn, Metric=c, **paired(g[g.Kind == "fPINN"].set_index("Seed")[c], g[g.Kind == "PINN"].set_index("Seed")[c], "fPINN - PINN")))
if len(cl):
    LOST_MM = 5.0  # loss of tracking: Cartesian error above 5 mm (all valid runs stay below 2 mm)
    lost = cl["Unstable"].astype(bool) | (cl["Max error (mm)"].fillna(np.inf) > LOST_MM)
    if lost.any(): print("Runs that lost tracking (excluded from the means):"); print(cl[lost][["Scenario", "Model", "Seed", "Speed scale", "Payload (kg)", "Max error (mm)", "Saturated (%)"]].to_string(index=False))
    ok = cl[~lost]
    print("\nClosed loop, Cartesian RMSE (µm), mean over seeds:")
    print((ok.groupby(["Scenario", "Speed scale", "Payload (kg)", "Model"])["Cartesian RMSE (mm)"].mean()*1e3).unstack("Model").round(1).to_string())
    for scn in SCENARIOS:
        for sc in SPEED_SCALES:
            for m in PAYLOADS:
                g = ok[(ok.Scenario == scn) & (ok["Speed scale"] == sc) & (ok["Payload (kg)"] == m)]
                a = g[g.Model == "fPINN"].set_index("Seed")["Cartesian RMSE (mm)"]; b = g[g.Model == "PINN"].set_index("Seed")["Cartesian RMSE (mm)"]
                gap = g[g.Model == "nominal"]["Cartesian RMSE (mm)"].mean() - g[g.Model == "oracle"]["Cartesian RMSE (mm)"].mean()
                row = dict(Scenario=scn + " closed loop", Metric=f"Cartesian RMSE {sc} s {m:.0f} kg (mm)", **paired(a, b, "fPINN - PINN"))
                for k, s_ in (("fPINN", a), ("PINN", b)):
                    row[f"gap closed {k} (%)"] = 100*(g[g.Model == "nominal"]["Cartesian RMSE (mm)"].mean() - s_.mean())/gap if gap else np.nan
                out.append(row)
summ = pd.DataFrame(out); summ.to_csv("NB08_fPINN_Summary" + SUF + ".csv", index=False)
print("\nPaired comparisons over seeds (negative mean diff = fPINN better):"); print(summ.round(5).to_string(index=False))

## Figure: test torque error per scenario and closed-loop error per speed

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
groups = [(s, k) for s in SCENARIOS for k in ("PINN", "fPINN")]
data = [1e3*df[(df.Scenario == s) & (df.Kind == k)]["Test RMSE (N m)"].values for s, k in groups]
ax[0].boxplot(data); ax[0].set_xticks(range(1, len(groups) + 1)); ax[0].set_xticklabels([f"{s}\n{k}" for s, k in groups], fontsize=8)
ax[0].set_ylabel("Test torque RMSE (mN m)"); ax[0].set_title("(a) multibody plant, open loop", loc="left")
if len(cl):
    g = ok[ok["Payload (kg)"] == PAYLOADS[0]].groupby(["Scenario", "Speed scale", "Model"])["Cartesian RMSE (mm)"].mean()*1e3
    models = ["nominal", "PINN", "fPINN", "oracle"]; xt = []; x0 = 0
    for scn in SCENARIOS:
        for sc in SPEED_SCALES:
            for k, mdl in enumerate(models):
                if (scn, sc, mdl) in g: ax[1].bar(x0 + 0.2*k, g[(scn, sc, mdl)], 0.2, color=f"C{k}", label=mdl if x0 == 0 else None)
            xt.append((x0 + 0.3, f"{scn}\n{sc}")); x0 += 1
    ax[1].set_xticks([x for x, _ in xt]); ax[1].set_xticklabels([l for _, l in xt], fontsize=7); ax[1].set_yscale("log")
    ax[1].set_ylabel("Cartesian RMSE (µm)"); ax[1].set_title(f"(b) CTC + I, {PAYLOADS[0]:.0f} kg", loc="left"); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig("fig_nb08_fpinn" + SUF + ".png", dpi=200); fig.savefig("fig_nb08_fpinn" + SUF + ".pdf"); plt.show()
print(f"total {(time.time()-t_all)/60:.0f} min")